# Electric Bass full song — Google Colab mandiri

Notebook ini menentukan ROOT sendiri. Checkout/runtime/bobot yang masih ada dipakai ulang.
Pada VM baru, upload **cuda_reference_*.zip** dari exporter Mac saat diminta (bukan ZIP hasil).
Runtime/bobot dipasang hanya jika belum tersedia; VM baru memerlukan unduhan model lagi.
Pilih GPU sebelum Run All. Lanjutkan upload audio asli dan helper jika source bundle lama belum menyertakannya.
Default generasi: satu konteks 60 detik. Sesudah berhasil, ubah DURATION menjadi None dan ulangi sel generasi saja untuk seluruh lagu, maksimal 600 detik.
Tidak menjalankan ulang baseline B atau A/B. DRY_RUN=True hanya memeriksa plan.


In [ ]:
from pathlib import Path
import importlib.util, json, shutil, subprocess, sys, uuid
DRY_RUN = False  # True = plan saja, tanpa instalasi/unduhan/inferensi
BUNDLE_ZIP = None  # Opsional path cuda_reference_*.zip yang sudah ada
ROOT_HINT = None  # Opsional Path ke source yang sudah ada pada VM
def discover_root(base, preferred=None):
    """Find a bridge checkout without notebook globals or recursively scanning model caches."""
    base = Path(base).resolve()
    markers = ("configs/assets-base.json", "tools/stem_eval/run.py", "scripts/setup_runtime.py")
    def valid(path):
        return all((path / name).is_file() for name in markers)
    if preferred is not None:
        path = Path(preferred).expanduser().resolve()
        if not valid(path):
            raise RuntimeError(f"ROOT yang dipilih bukan checkout bridge: {path}")
        return path
    candidates = set()
    for folder in dict.fromkeys((base, base.parent)):
        for path in (folder, folder / "source", folder / "StemsGenerating-Models",
                     *folder.glob("songmap_cuda_*/source")):
            if valid(path): candidates.add(path.resolve())
    ready = [path for path in candidates
             if (path / "vendor/ACE-Step-1.5/.venv/bin/python").is_file()
             and (path / "vendor/ACE-Step-1.5/checkpoints/acestep-v15-base/model.safetensors").is_file()]
    choices = sorted(ready or candidates)
    if len(choices) > 1:
        raise RuntimeError("Ada beberapa checkout bridge. Isi ROOT_HINT dengan salah satu path ini: "
                           + ", ".join(map(str, choices)))
    return choices[0] if choices else None

ROOT = discover_root(Path.cwd(), ROOT_HINT)
if ROOT is None and Path.cwd() != Path("/content") and Path("/content").is_dir():
    ROOT = discover_root(Path("/content"))
if ROOT is None:
    from pathlib import Path, PurePosixPath
    import hashlib, json, shutil, stat, subprocess, uuid, zipfile
    if not DRY_RUN:
        from google.colab import files
        assert shutil.which("nvidia-smi"), "Pilih Runtime → Change runtime type → GPU, lalu jalankan ulang."
    if BUNDLE_ZIP is None:
        from google.colab import files
        uploaded = files.upload()
        assert len(uploaded) == 1, "Upload tepat satu cuda_reference_*.zip hasil ekspor Mac"
        BUNDLE_ZIP = Path(next(iter(uploaded))).resolve()
    assert BUNDLE_ZIP.suffix.lower() == ".zip", "Pilih ZIP hasil ekspor Mac"
    with zipfile.ZipFile(BUNDLE_ZIP) as archive:
        entries = archive.infolist()
        names = [entry.filename for entry in entries]
        assert len(names) == len(set(names)), "Paket memiliki nama file ganda"
        assert sum(entry.file_size for entry in entries) <= 256*1024**2, "Paket terlalu besar; gunakan exporter tanpa bobot"
        assert "bundle.json" in names, "Gunakan cuda_reference_*.zip hasil exporter, bukan feedback.zip"
        bundle_bytes = archive.read("bundle.json")
        manifest = json.loads(bundle_bytes)
        assert manifest.get("schema_version") == 1, "Paket CUDA tidak didukung"
        expected = {record["path"] for record in manifest["files"]} | {"bundle.json"}
        assert set(names) == expected, "Isi ZIP tidak cocok dengan manifest paket CUDA"
        destination_key = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
        COLAB_BUNDLE = (Path.cwd() / ("songmap_cuda_" + destination_key)).resolve()
        data_to_write = {}
        for entry in entries:
            relative = PurePosixPath(entry.filename)
            assert not relative.is_absolute() and ".." not in relative.parts and "\\" not in entry.filename, "Path ZIP tidak valid"
            assert not stat.S_ISLNK(entry.external_attr >> 16), "Paket tidak boleh memuat symlink"
            target = (COLAB_BUNDLE / entry.filename).resolve()
            assert target.is_relative_to(COLAB_BUNDLE), "Path paket keluar dari folder tujuan"
        for record in manifest["files"]:
            data = archive.read(record["path"])
            assert hashlib.sha256(data).hexdigest() == record["sha256"], f"Paket rusak: {record['path']}"
            data_to_write[record["path"]] = data
        data_to_write["bundle.json"] = bundle_bytes
        for name, data in data_to_write.items():
            target = COLAB_BUNDLE / name
            if target.exists():
                assert target.is_file() and hashlib.sha256(target.read_bytes()).digest() == hashlib.sha256(data).digest(), f"Snapshot lokal berubah: {name}"
        COLAB_BUNDLE.mkdir(parents=True, exist_ok=True)
        for name, data in data_to_write.items():
            target = COLAB_BUNDLE / name
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
    print("Paket diverifikasi:", COLAB_BUNDLE)
    print("Commit snapshot:", manifest["bridge_commit"])
    print("Input B SHA256:", manifest["source_sha256"])
    ROOT = COLAB_BUNDLE / "source"
print("Checkout bridge:", ROOT)


In [ ]:
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT / "tools/stem_eval")]
from cuda_reference import check_cuda, verify_bundle
from runtime_environment import runtime_environment
from stream_process import run_logged
from system_tools import require_audio_tools
if (ROOT.parent / "bundle.json").is_file():
    verify_bundle(ROOT.parent)
MODEL_PYTHON = ROOT / "vendor/ACE-Step-1.5/.venv/bin/python"
UPSTREAM = ROOT / "vendor/ACE-Step-1.5"
ASSETS = ROOT / "configs/assets-base.json"
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Pilih Runtime → Change runtime type → GPU"
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], text=True))
    if any(shutil.which(name) is None for name in ("ffmpeg", "ffprobe")):
        run_logged(["apt-get", "update"], check=True)
        run_logged(["apt-get", "install", "-y", "ffmpeg"], check=True)
require_audio_tools()
missing = sum(item["size"] for item in json.loads(ASSETS.read_text())["files"] if not (UPSTREAM / item["destination"]).is_file())
required = 2 * missing + 5 * 1024**3 + (8 * 1024**3 if not MODEL_PYTHON.is_file() else 0)
print("Ruang kosong GiB:", round(shutil.disk_usage(ROOT).free/1024**3, 1), "| Estimasi ruang baru GiB:", round(required/1024**3, 1))
if not DRY_RUN:
    assert shutil.disk_usage(ROOT).free >= required, "Ruang disk belum cukup untuk runtime/cache dan bobot"


## Runtime model terisolasi (Python 3.11)


In [ ]:
if DRY_RUN:
    print("DRY RUN: setup runtime dilewati")
else:
    environment = runtime_environment(ROOT)
    needs_setup = not MODEL_PYTHON.is_file()
    if not needs_setup:
        probe = subprocess.run([str(MODEL_PYTHON), "-c", "import torch,torchaudio,transformers,acestep.handler"], env=environment, capture_output=True, text=True)
        needs_setup = probe.returncode != 0
    if needs_setup:
        import venv
        bootstrap = ROOT / ".bootstrap"
        bootstrap_python = bootstrap / "bin/python"
        if not bootstrap_python.is_file():
            venv.EnvBuilder(with_pip=False).create(bootstrap)
        # Support source bundles with the older ensurepip-based installer too.
        run_logged([sys.executable, "-m", "pip", "--python", str(bootstrap_python), "install", "pip", "uv==0.10.0"], env=environment, check=True)
        run_logged([sys.executable, str(ROOT / "scripts/setup_runtime.py"), "--install"], env=environment, check=True)
    else:
        print("Memakai runtime model yang sudah tersedia")
    print(json.dumps(check_cuda(MODEL_PYTHON), indent=2))


## Bobot model


In [ ]:
if DRY_RUN:
    print("DRY RUN: unduhan bobot dilewati")
else:
    records = json.loads(ASSETS.read_text())["files"]
    available = all((UPSTREAM / item["destination"]).is_file() and (UPSTREAM / item["destination"]).stat().st_size == item["size"] for item in records)
    if available:
        print("Semua file bobot tersedia. Hash diverifikasi lagi saat generasi.")
    else:
        run_logged([str(MODEL_PYTHON), str(ROOT / "scripts/model_assets.py"), "--manifest", str(ASSETS), "--upstream", str(UPSTREAM), "--download"], env=runtime_environment(ROOT), check=True)


## Audio dan helper
Upload yang masih ada di memori/disk dipakai kembali. Jika helper sudah ada di paket, cukup upload audio.


In [ ]:
import importlib.util, uuid
from pathlib import Path
from google.colab import files


# Pakai kembali upload dari sel yang gagal, atau file yang masih ada di VM.
def input_names(payload):
    scripts = [n for n in payload if Path(n).name.startswith("electric_full_song") and Path(n).suffix == ".py"]
    audios = [n for n in payload if Path(n).suffix.lower() in {".wav", ".mp3", ".m4a", ".flac", ".aac", ".aif", ".aiff", ".ogg"}]
    return scripts, audios
uploaded = dict(globals().get("uploaded") or {})
packaged_helper = ROOT / "scripts/electric_full_song.py"
if not input_names(uploaded)[0] and packaged_helper.is_file():
    uploaded["electric_full_song.py"] = packaged_helper.read_bytes()
scripts, audios = input_names(uploaded)
if len(scripts) != 1 or len(audios) != 1:
    local_helpers = list(Path.cwd().glob("electric_full_song*.py"))
    local_audio = [p for p in Path.cwd().iterdir() if p.is_file() and p.suffix.lower() in {".wav", ".mp3", ".m4a", ".flac", ".aac", ".aif", ".aiff", ".ogg"}]
    helper_file = local_helpers[0] if len(local_helpers) == 1 else (packaged_helper if packaged_helper.is_file() else None)
    if helper_file is not None and len(local_audio) == 1:
        uploaded = {helper_file.name: helper_file.read_bytes(), local_audio[0].name: local_audio[0].read_bytes()}
    else:
        uploaded = files.upload()
        if not input_names(uploaded)[0] and packaged_helper.is_file():
            uploaded["electric_full_song.py"] = packaged_helper.read_bytes()
scripts, audios = input_names(uploaded)
assert len(scripts) == 1 and len(audios) == 1, "Pilih satu helper electric_full_song.py dan satu audio asli"
tools_dir = ROOT / "runs" / ("full_song_tools_" + uuid.uuid4().hex[:12])
tools_dir.mkdir(parents=True)
helper = tools_dir / "electric_full_song.py"
helper.write_bytes(uploaded[scripts[0]])
SOURCE_PATH = tools_dir / ("input" + Path(audios[0]).suffix.lower())
SOURCE_PATH.write_bytes(uploaded[audios[0]])
spec = importlib.util.spec_from_file_location("electric_single_pass", helper)
fullsong = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fullsong)
print("ROOT:", ROOT)
print("Audio siap:", SOURCE_PATH)


## Satu panggilan: 60 detik, lalu seluruh lagu
Ganti DURATION menjadi None setelah hasil 60 detik dinilai. VAE tiling internal berbeda dari chunk musik independen.


In [ ]:
from IPython.display import Audio, display

DURATION = 60  # None = seluruh file, maksimum 600 detik
START = 0
GENERATE = not DRY_RUN
FULL_OUT = ROOT / "runs" / ("electric_single_pass_" + uuid.uuid4().hex[:12])
report = fullsong.generate(
    ROOT, SOURCE_PATH, FULL_OUT,
    start=START, duration=DURATION, execute=GENERATE,
)
print("Status:", report["status"], "| Folder:", FULL_OUT)
if GENERATE:
    print("Peak VRAM allocated (GiB):", report["peak_cuda_allocated_bytes"] / 2**30)
    for name in ("electric_preview.mp3", "mix_preview.mp3"):
        print(name)
        display(Audio(filename=str(FULL_OUT / name), normalize=False))


## Unduh WAV dan paket feedback
Jalankan sendiri setelah error untuk mengambil report yang tersedia. Simpan hasil sebelum sesi berakhir.


In [ ]:
for name in ("full_song_result.zip", "full_song_feedback.zip"):
    if (FULL_OUT / name).is_file():
        files.download(str(FULL_OUT / name))
